In [5]:


import tensorflow as tf
from tensorflow.keras import models, layers, callbacks
import random
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import pathlib



img_size = 224
batch_size = 32
data_dir = pathlib.Path("/home/sir-croco/Documents/Data/")

class_names = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
print("Categories (check the order):", class_names)

allowed_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp'}
image_files = sorted(
    str(p) for p in data_dir.rglob('*')
    if p.is_file() and p.suffix.lower() in allowed_exts
)

if not image_files:
    raise ValueError(f"No image files found under {data_dir}")

if len(image_files) > 12000:
    random.seed(123)
    image_files = random.sample(image_files, 12000)


train_paths, val_paths = train_test_split(
    image_files, 
    test_size=0.2, 
    random_state=123, 
    stratify=[pathlib.Path(p).parent.name for p in image_files] 
)


train_files = tf.data.Dataset.from_tensor_slices(train_paths)
val_files = tf.data.Dataset.from_tensor_slices(val_paths)

def load_and_preprocess(file_path):
    parts = tf.strings.split(file_path, '/')
    label = tf.cast(tf.equal(parts[-2], class_names[1]), tf.int32)

    image = tf.io.read_file(file_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize_with_pad(image, img_size, img_size)  
    image = tf.cast(image, tf.float32)

    return image, label

train_dataset = (train_files
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000, seed=123)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE))

val_dataset = (val_files
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE))

base_model= tf.keras.applications.ConvNeXtSmall(

    input_shape= (224,224,3),
    include_top= False,
    weights= "imagenet",
)

base_model.trainable= False

data_aug= models.Sequential([
    
    layers.RandomContrast(0.1),        
    layers.RandomBrightness(0.1)
])

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_aug,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation=None)
])

early_stop = EarlyStopping(
    monitor='val_loss',         
    patience=6,                 
    restore_best_weights=True,  
    verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,             # Cut LR by 80% when it plateaus
    patience=4,             # Wait 4 epochs of no improvement before cutting LR
    min_lr=1e-7,            # Prevent LR from dropping too low
    verbose=1
)

model.compile(
    optimizer= tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics= ['accuracy']
)

history_1_base= model.fit(
    train_dataset,
    validation_data= val_dataset,
    epochs= 25,
    callbacks=[early_stop, reduce_lr] ,
    batch_size=64,
    shuffle=True
)

Categories (check the order): ['legit', 'scam']
Epoch 1/25


/home/sir-croco/.myenv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1790465115.039719  563916 service.cc:153] XLA service 0x7261d8070430 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790465115.039736  563916 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.26.0)
I0000 00:00:1790465115.341701  563916 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790465116.923521  563916 cuda_dnn.cc:461] Loaded cuDNN version 92600
I0000 00:00:1790465117.949743  564070 subprocess_compilation.cc:348] pt

  1/300 ━━━━━━━━━━━━━━━━━━━━ 1:42:07 20s/step - accuracy: 0.4375 - loss: 1.4585

I0000 00:00:1790465127.228390  563916 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


300/300 ━━━━━━━━━━━━━━━━━━━━ 61s 134ms/step - accuracy: 0.4736 - loss: 0.8033 - val_accuracy: 0.4896 - val_loss: 0.6705 - learning_rate: 1.0000e-04
Epoch 2/25
300/300 ━━━━━━━━━━━━━━━━━━━━ 36s 116ms/step - accuracy: 0.5742 - loss: 0.6566 - val_accuracy: 0.6338 - val_loss: 0.5734 - learning_rate: 1.0000e-04
Epoch 3/25
300/300 ━━━━━━━━━━━━━━━━━━━━ 36s 116ms/step - accuracy: 0.6608 - loss: 0.5790 - val_accuracy: 0.7042 - val_loss: 0.5102 - learning_rate: 1.0000e-04
Epoch 4/25
300/300 ━━━━━━━━━━━━━━━━━━━━ 36s 116ms/step - accuracy: 0.7205 - loss: 0.5216 - val_accuracy: 0.7667 - val_loss: 0.4629 - learning_rate: 1.0000e-04
Epoch 5/25
300/300 ━━━━━━━━━━━━━━━━━━━━ 36s 116ms/step - accuracy: 0.7554 - loss: 0.4813 - val_accuracy: 0.7800 - val_loss: 0.4303 - learning_rate: 1.0000e-04
Epoch 6/25
300/300 ━━━━━━━━━━━━━━━━━━━━ 36s 117ms/step - accuracy: 0.7771 - loss: 0.4494 - val_accuracy: 0.8017 - val_loss: 0.4032 - learning_rate: 1.0000e-04
Epoch 7/25
300/300 ━━━━━━━━━━━━━━━━━━━━ 36s 117ms/step - 

In [6]:


base_model.trainable = True

for layer in base_model.layers:

    if 'stage_3' in layer.name:

        if isinstance(layer, (tf.keras.layers.BatchNormalization, tf.keras.layers.LayerNormalization)):
            layer.trainable = False
        else:
            layer.trainable = True
    else:

        layer.trainable = False



model.compile(
    optimizer= tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics= ['accuracy']
)

early_stop_fine = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,            
    patience=4,             
    min_lr=1e-7,           
    verbose=1
)

history_2= model.fit(
    train_dataset,
    validation_data= val_dataset,
    epochs= 20,
    callbacks=[early_stop_fine,reduce_lr] ,
    batch_size= 32,
    shuffle=True,

)

Epoch 1/20


I0000 00:00:1790466170.640007  563915 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_119723__.632
I0000 00:00:1790466171.405732  569137 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_442', 8480 bytes spill stores, 8508 bytes spill loads

I0000 00:00:1790466172.036644  569141 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_1_278', 136 bytes spill stores, 104 bytes spill loads

I0000 00:00:1790466172.719876  569141 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_1_280', 136 bytes spill stores, 104 bytes spill loads

E0000 00:00:1790466173.100489  563915 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


300/300 ━━━━━━━━━━━━━━━━━━━━ 65s 146ms/step - accuracy: 0.8791 - loss: 0.2655 - val_accuracy: 0.8925 - val_loss: 0.2281 - learning_rate: 5.0000e-06
Epoch 2/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 40s 129ms/step - accuracy: 0.8944 - loss: 0.2371 - val_accuracy: 0.9000 - val_loss: 0.2087 - learning_rate: 5.0000e-06
Epoch 3/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 40s 129ms/step - accuracy: 0.9036 - loss: 0.2168 - val_accuracy: 0.9054 - val_loss: 0.1956 - learning_rate: 5.0000e-06
Epoch 4/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 40s 129ms/step - accuracy: 0.9132 - loss: 0.1998 - val_accuracy: 0.9175 - val_loss: 0.1814 - learning_rate: 5.0000e-06
Epoch 5/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 40s 129ms/step - accuracy: 0.9195 - loss: 0.1880 - val_accuracy: 0.9250 - val_loss: 0.1728 - learning_rate: 5.0000e-06
Epoch 6/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 39s 125ms/step - accuracy: 0.9230 - loss: 0.1788 - val_accuracy: 0.9242 - val_loss: 0.1666 - learning_rate: 5.0000e-06
Epoch 7/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 40s 130ms/step - 

In [16]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

required_names = ("model", "val_paths", "load_and_preprocess", "class_names")
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        f"Missing {', '.join(missing_names)}. Run the data setup and training cells first."
    )
if len(class_names) != 2:
    raise ValueError(f"Expected exactly 2 classes, found: {class_names}")
if not val_paths:
    raise ValueError("The validation split is empty.")

# Keep evaluation batches small to reduce GPU memory pressure.
eval_batch_size = 4
eval_dataset = (
    tf.data.Dataset.from_tensor_slices(val_paths)
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(eval_batch_size)
    .prefetch(1)
)

y_logits = model.predict(eval_dataset, verbose=0).reshape(-1)
y_true = np.concatenate(
    [labels.numpy().reshape(-1) for _, labels in eval_dataset]
).astype(int)
if y_true.size == 0:
    raise ValueError("No validation examples were evaluated.")

# The model emits logits, so convert them to probabilities before thresholding.
y_pred_prob = tf.nn.sigmoid(y_logits).numpy()
threshold = 0.5
y_pred = (y_pred_prob >= threshold).astype(int)

accuracy = accuracy_score(y_true, y_pred)
balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

print(f"Validation examples: {len(y_true)}")
print(f"Threshold (positive = {class_names[1]}): {threshold:.2f}")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_accuracy:.4f}")
print(f"Precision ({class_names[1]}): {precision:.4f}")
print(f"Recall ({class_names[1]}):    {recall:.4f}")
print(f"F1 ({class_names[1]}):        {f1:.4f}")
if np.unique(y_true).size == 2:
    print(f"ROC-AUC:           {roc_auc_score(y_true, y_pred_prob):.4f}")
else:
    print("ROC-AUC: unavailable because validation contains only one class")

print("\nConfusion matrix (rows = actual, columns = predicted):")
print(f"Classes: {class_names}")
print(cm)

Validation examples: 2400
Threshold (positive = scam): 0.50
Accuracy:          0.9379
Balanced accuracy: 0.9380
Precision (scam): 0.9459
Recall (scam):    0.9312
F1 (scam):        0.9385
ROC-AUC:           0.9878

Confusion matrix (rows = actual, columns = predicted):
Classes: ['legit', 'scam']
[[1114   65]
 [  84 1137]]


In [21]:
import tensorflow as tf

def predict_single_image(image_path, model, class_names=['legit', 'scam']):
    # قراءة الصورة وإعادة ضبط الحجم
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize_with_pad(image, 224, 224)
    image = tf.cast(image, tf.float32)
    
    # إضافة بعد الدفعة ("batch dimension")
    image_batch = tf.expand_dims(image, axis=0)
    
    # حساب الاحتمالية عبر دالة ("sigmoid")
    logits = model(image_batch, training=False)
    scam_probability = tf.nn.sigmoid(logits).numpy()[0][0]
    
    # ربط الفهارس: Index 0 = "legit", Index 1 = "scam"
    if scam_probability >= 0.65:
        class_index = 1
        confidence = scam_probability
    else:
        class_index = 0
        confidence = 1.0 - scam_probability
        
    predicted_label = class_names[class_index]
    
    return predicted_label, class_index, float(confidence)

# تشغيل التوقع
inp = input("Enter image path: ").strip().strip('"').strip("'")
label, idx, conf = predict_single_image(inp, model, class_names)

print(f"Predicted Class: {label} ")
print(f"Confidence: {conf * 100:.2f}%")

Predicted Class: legit 
Confidence: 57.30%


In [22]:
# حفظ النموذج بالكامل في ملف واحد
model.save("scam_classifier_model.keras")

In [23]:
model.save_weights("model_weights.weights.h5")